# ADMM Distributed Stochastic Coordination Benchmark

This notebook implements the paper-revision ADMM comparison baseline.

The goal is not to replace the current centralized stochastic coordination method. The goal is to solve the same underlying stochastic coordination problem with an iterative distributed method, so that runtime, iterations, communication burden, convergence behavior, and scenario-dependent price recovery can be compared.

Comparison narrative:

- Current method: offline centralized stochastic optimization -> direct dual-price extraction -> one-shot decentralized operation.
- ADMM benchmark: DER-local stochastic optimization -> coordinator consensus projection -> iterative dual/price updates -> convergence-based decentralized coordination.

In this main `admm` folder, both the centralized benchmark and the ADMM benchmark include the original small objective regularization term.

## Coupling Decoupling Flow

The centralized model in `0527.ipynb` has DER-local constraints and one explicit market-level coupling constraint.

DER-local variables and constraints:

- renewable scenario: `R[i,t,s]`
- day-ahead commitment: `x[i,t]`
- real-time sell/buy: `yp[i,t,s]`, `ym[i,t,s]`
- internal sell/buy: `dp[i,t,s]`, `dm[i,t,s]`
- storage state/charge/discharge: `z`, `zc`, `zd`

The coupling constraint is the internal market balance:

```text
For every time t and scenario s:

    sum_i dp_i(t,s) = sum_i dm_i(t,s)
```

For ADMM, we use local net internal trade:

```text
q_i(t,s) = dp_i(t,s) - dm_i(t,s)
```

The coordinator owns consensus copies `consensus_q_i(t,s)` and enforces:

```text
q_i(t,s) = consensus_q_i(t,s)
sum_i consensus_q_i(t,s) = 0
```

Iteration flow:

1. Each DER solves its own local stochastic QP over all scenarios.
2. Each DER sends the full matrix `q_i(t,s)` to the coordinator.
3. The coordinator projects `q_i + u_i` onto the constraint `sum_i consensus_q_i(t,s)=0` for every `(t,s)`.
4. The scaled dual variable `u_i` is updated.
5. Scenario-dependent internal prices are recovered from the ADMM duals.

Price convention follows `0527.ipynb`:

```text
lambda_admm(t,s) = rho * mean_i u_i(t,s)
P_internal_admm(t,s) = -S * lambda_admm(t,s)
```

One ADMM iteration updates all scenario price vectors together, not one scenario at a time.

## ADMM Local Problem

Each DER solves a local ADMM subproblem. In words, the DER maximizes its own stochastic profit while staying close to the coordinator consensus target from the previous iteration.

Plain-text form:

```text
maximize over DER-local variables:

    local_market_profit_i
  - objective_regularization_i
  - (rho / 2) * || q_i - consensus_q_i_previous + u_i_previous ||^2

where:

    q_i(t,s) = dp_i(t,s) - dm_i(t,s)
```

The DER-local market profit is:

```text
sum_t P_DA[t] * x_i[t]
+ average_s sum_t (P_RT[t,s] * yp_i[t,s] - P_PN[t,s] * ym_i[t,s])
```

In this main `admm` folder, ADMM also keeps the original small objective regularization term:

```text
objective_regularization_i
= eps / S * sum_{t,s} (dp_i[t,s]^2 + dm_i[t,s]^2)
```

Storage dynamics and renewable balance remain local. The only market-level coupling is handled through ADMM consensus on `q_i=dp_i-dm_i`.

## How the ADMM Implementation Works

The implementation is in `admm_stochastic_benchmark.py`. The most important functions/classes are:

```text
solve_centralized(...)
    Builds and solves the centralized stochastic benchmark.
    This is the reference solution and source of centralized dual prices.

LocalDERModel
    Stores one DER-local Gurobi model.
    Each DER has variables x, yp, ym, dp, dm, z, zc, zd.

LocalDERModel.set_admm_objective(target)
    Rebuilds the local objective at every ADMM iteration.
    target is consensus_q_i - u_i from the previous coordinator step.

project_internal_balance_consensus(w)
    Coordinator update.
    Projects w_i(t,s)=q_i(t,s)+u_i(t,s) onto sum_i consensus_q_i(t,s)=0.

run_admm(...)
    Main ADMM loop: local solve, coordinator projection, dual update,
    residual calculation, convergence check, and price recovery.
```

Detailed iteration logic:

```text
Initialize:
    q_i(t,s) = 0
    consensus_q_i(t,s) = 0
    u_i(t,s) = 0

For k = 1, 2, ...:

    1. Local DER solve
       Each DER receives target_i = consensus_q_i - u_i.
       It solves its stochastic QP and returns:

           dp_i(t,s), dm_i(t,s)
           q_i(t,s) = dp_i(t,s) - dm_i(t,s)

    2. Coordinator consensus update
       Coordinator forms:

           w_i(t,s) = q_i(t,s) + u_i(t,s)

       Then it projects onto internal market balance:

           consensus_q_i(t,s) = w_i(t,s) - mean_j w_j(t,s)

       This makes:

           sum_i consensus_q_i(t,s) = 0

    3. Dual update
       The scaled dual is updated by:

           u_i(t,s) = u_i(t,s) + q_i(t,s) - consensus_q_i(t,s)

    4. Price recovery
       The scenario-dependent coordination price is recovered as:

           lambda_admm(t,s) = rho * mean_i u_i(t,s)
           P_internal_admm(t,s) = -S * lambda_admm(t,s)

    5. Residuals
       Primal residual measures local-consensus disagreement:

           primal_residual = norm(q - consensus_q)

       Dual residual measures coordinator-copy movement:

           dual_residual = rho * norm(consensus_q - consensus_q_previous)

    6. Stop if both residuals are below tolerance.
```

Economic interpretation:

- `q_i(t,s) > 0` means DER `i` is a net internal seller in scenario `(t,s)`.
- `q_i(t,s) < 0` means DER `i` is a net internal buyer.
- The coordinator does not optimize DER storage or market decisions directly.
- The coordinator only enforces the market-clearing condition through consensus.
- The ADMM dual variable becomes the internal market coordination price.

Scenario handling:

- Every local DER problem includes all scenarios at once.
- Every ADMM iteration updates the full `T x S` price matrix together.
- ADMM is not run separately for each scenario.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name in ["admm", "admm_noreg"]:
    ROOT = ROOT.parent
MODULE_DIR = ROOT / "admm"

# Force the notebook to load the local module from this folder, not a stale
# cached copy from a previous kernel state or the sibling admm/admm_noreg folder.
for key in ["admm_stochastic_benchmark"]:
    if key in sys.modules:
        del sys.modules[key]
importlib.invalidate_caches()

sys.path = [
    str(MODULE_DIR),
    str(ROOT),
] + [p for p in sys.path if p not in [str(MODULE_DIR), str(ROOT)]]

from admm_stochastic_benchmark import (
    build_problem_data,
    solve_centralized,
    run_admm,
    compare_centralized_admm,
    run_scalability_grid,
    run_overnight_scalability,
    summarize_scalability_results,
    run_rho_sensitivity,
    run_full_comparison_case,
    build_full_comparison_from_solutions,
)

pd.options.display.float_format = "{:.6f}".format
plt.rcParams["figure.figsize"] = (10, 4)

print("Loaded module from:", MODULE_DIR / "admm_stochastic_benchmark.py")

## Experiment Settings

Default settings use 10 DERs and 100 scenarios. For a quick sanity check, reduce `S` and `MAX_ITER` before running the full experiment.

The penalty parameter `rho` controls ADMM consensus enforcement. In the small smoke tests, `rho=1.0` reduced the internal balance residual more reliably than very small values.

In [ ]:
N_DER = 10
S = 100
SEED = 1
LEVEL = "high"
EPS = 1e-8
RHO = 1.0
MAX_ITER = 1000
ABS_TOL = 1e-3
REL_TOL = 1e-4

# Set to 1 if you want to see Gurobi solver output.
OUTPUT_FLAG = 0

In [ ]:
data = build_problem_data(
    n_der=N_DER,
    scenarios=S,
    seed=SEED,
    level=LEVEL,
)

print(f"Loaded I={data['I']}, T={data['T']}, S={data['S']}")
print("Selected DER files:", data["selected_files"])
print("R shape:", data["R"].shape)
print("P_RT shape:", data["P_RT"].shape)

## 1. Centralized Stochastic Benchmark

This cell solves the centralized continuous stochastic QP corresponding to the current model in `0527.ipynb`.

Key outputs:

- coupling constraint: `sum_i dp[i,t,s] == sum_i dm[i,t,s]`
- dual variable: `lambda_dual[t,s]`
- internal price: `P_internal[t,s] = -S * lambda_dual[t,s]`

In [ ]:
centralized = solve_centralized(
    data,
    eps=EPS,
    time_limit=1200,
    output_flag=OUTPUT_FLAG,
)

print("Centralized expected profit:", centralized["expected_profit"])
print("Centralized regularized profit:", centralized["regularized_profit"])
print("Centralized total seconds:", centralized["total_seconds"])
print("Max balance residual:", np.max(np.abs(centralized["balance_residual"])))

## 2. ADMM Distributed Benchmark

In each ADMM iteration, every DER solves a local stochastic optimization problem. The coordinator then performs the consensus projection for all `(t,s)` entries at once.

The communication burden is approximated as:

```text
communication_scalars = 2 * iteration * I * T * S
```

This counts one DER-to-coordinator transfer of `q_i(t,s)` and one coordinator-to-DER transfer of updated consensus or price information per iteration.

In [ ]:
admm = run_admm(
    data,
    rho=RHO,
    eps=EPS,
    max_iter=MAX_ITER,
    abs_tol=ABS_TOL,
    rel_tol=REL_TOL,
    output_flag=OUTPUT_FLAG,
    verbose=True,
)

print("ADMM converged:", admm["converged"])
print("ADMM iterations:", admm["iterations"])
print("ADMM expected profit:", admm["expected_profit"])
print("ADMM total seconds:", admm["total_seconds"])
print("ADMM max balance residual:", np.max(np.abs(admm["balance_residual"])))

## 3. Centralized vs ADMM Summary

The reported optimality gap is:

```text
optimality_gap = centralized_expected_profit - admm_expected_profit
relative_gap = optimality_gap / abs(centralized_expected_profit)
```

If ADMM has not converged, its expected profit can appear better than the centralized benchmark because the internal balance constraint may still be violated. Always interpret the gap together with `max_balance_error`, `primal_residual`, and `dual_residual`.

In [ ]:
summary = compare_centralized_admm(centralized, admm)
pd.DataFrame([summary]).T.rename(columns={0: "value"})

In [ ]:
hist = admm["history"].copy()
hist.tail(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(hist["iteration"], hist["primal_residual"], label="primal")
axes[0].semilogy(hist["iteration"], hist["dual_residual"], label="dual")
axes[0].semilogy(hist["iteration"], hist["eps_primal"], "--", label="eps primal")
axes[0].semilogy(hist["iteration"], hist["eps_dual"], "--", label="eps dual")
axes[0].set_title("ADMM residuals")
axes[0].set_xlabel("iteration")
axes[0].legend()

axes[1].plot(hist["iteration"], hist["objective"], label="ADMM")
axes[1].axhline(centralized["expected_profit"], color="black", linestyle="--", label="centralized")
axes[1].set_title("Expected profit")
axes[1].set_xlabel("iteration")
axes[1].legend()

axes[2].semilogy(hist["iteration"], hist["max_balance_error"], label="max |sum dp - sum dm|")
axes[2].semilogy(hist["iteration"], hist["mean_abs_balance_error"], label="mean abs")
axes[2].set_title("Internal balance error")
axes[2].set_xlabel("iteration")
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Scenario-Dependent Internal Price Comparison

This section compares centralized dual prices and ADMM dual-update prices for a selected scenario.

Both use the same price convention:

```text
P_internal(t,s) = -S * lambda(t,s)
```

In [ ]:
s_fixed = min(0, S - 1)
price_df = pd.DataFrame({
    "t": np.arange(data["T"]),
    "P_RT": data["P_RT"][:, s_fixed],
    "P_PN": data["P_PN"][:, s_fixed],
    "centralized_internal_price": centralized["internal_price"][:, s_fixed],
    "admm_internal_price": admm["internal_price"][:, s_fixed],
})
price_df

## 5. Single-Seed 10-DER Run

Use this section when you want a quick paper-scale sanity check without launching the full overnight grid.

Default setup:

```text
N = 10
S = 100
seed = 1
```

This solves one centralized benchmark and one ADMM benchmark, then reports objective gap, runtime, ADMM iterations, communication burden, balance error, and price RMSE. The overnight section below is unchanged and can still be used for the full 50-case experiment.

In [ ]:
SINGLE_N_DER = 10
SINGLE_SCENARIOS = 100
SINGLE_SEED = 1
SINGLE_RHO = RHO
SINGLE_MAX_ITER = MAX_ITER

print("Single-case setup")
print("N:", SINGLE_N_DER)
print("S:", SINGLE_SCENARIOS)
print("seed:", SINGLE_SEED)
print("rho:", SINGLE_RHO)
print("max_iter:", SINGLE_MAX_ITER)

In [ ]:
# Reuse CAGG and ADMM results if you already ran the earlier cells.
# This cell only solves the additional individual-participation and DAgg replay models.

can_reuse_prior = (
    "data" in globals()
    and "centralized" in globals()
    and "admm" in globals()
    and data.get("I") == SINGLE_N_DER
    and data.get("S") == SINGLE_SCENARIOS
    and data.get("seed") == SINGLE_SEED
)

if can_reuse_prior:
    print("Reusing existing data, centralized, and admm results from earlier cells.")
    single_results = build_full_comparison_from_solutions(
        data=data,
        centralized=centralized,
        admm=admm,
        eps=EPS,
        time_limit=1200,
        output_flag=OUTPUT_FLAG,
    )
else:
    print("Existing CAGG/ADMM results do not match the single-case settings; solving full case once.")
    single_results = run_full_comparison_case(
        n_der=SINGLE_N_DER,
        scenarios=SINGLE_SCENARIOS,
        seed=SINGLE_SEED,
        rho=SINGLE_RHO,
        max_iter=SINGLE_MAX_ITER,
        eps=EPS,
        level=LEVEL,
        time_limit=1200,
        output_flag=OUTPUT_FLAG,
        verbose=True,
    )

single_data = single_results["data"]
single_individual = single_results["individual"]
single_centralized = single_results["centralized"]
single_dagg = single_results["dagg"]
single_admm = single_results["admm"]
single_admm_replay = single_results["admm_replay"]
single_aggregate_df = single_results["aggregate_df"]
single_individual_df = single_results["individual_df"]
single_price_summary_df = single_results["price_df"]

single_cagg_admm_summary = compare_centralized_admm(single_centralized, single_admm)
single_cagg_admm_summary.update({
    "n_der": SINGLE_N_DER,
    "scenarios": SINGLE_SCENARIOS,
    "seed": SINGLE_SEED,
    "rho": SINGLE_RHO,
    "eps": EPS,
})

pd.DataFrame([single_cagg_admm_summary]).T.rename(columns={0: "value"})

### Aggregate Profit Comparison

This table compares aggregate outcomes for:

- independent market participation,
- centralized aggregation (`CAGG`),
- decentralized aggregation replay using centralized prices (`DAgg`),
- ADMM final iterate, and ADMM price replay using the final ADMM price.

For `DAgg`, `ADMM final iterate`, and `ADMM price replay`, `profit_with_imbalance` includes the same imbalance adjustment convention used in `0527.ipynb`.

In [ ]:
display(single_aggregate_df)

plot_agg = single_aggregate_df.set_index("case")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_agg["profit_with_imbalance"].plot(kind="bar", ax=axes[0])
axes[0].set_title("Aggregate profit with imbalance adjustment")
axes[0].set_ylabel("profit")
axes[0].grid(axis="y", alpha=0.3)

plot_agg["imbalance_quantity_mwh"].plot(kind="bar", ax=axes[1], color="tab:orange")
axes[1].set_title("Aggregate imbalance quantity")
axes[1].set_ylabel("MWh")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Price Comparison

`DAgg` uses the centralized dual price directly. `ADMM final iterate` uses the final dual-updated price matrix from ADMM. `ADMM price replay` fixes that final ADMM price and resolves the decentralized DER replay problem.

In [ ]:
display(single_price_summary_df)

single_s_fixed = 0
single_price_df = pd.DataFrame({
    "t": np.arange(single_data["T"]),
    "P_RT": single_data["P_RT"][:, single_s_fixed],
    "P_PN": single_data["P_PN"][:, single_s_fixed],
    "cagg_internal_price": single_centralized["internal_price"][:, single_s_fixed],
    "dagg_internal_price": single_dagg["internal_price"][:, single_s_fixed],
    "admm_final_price": single_admm["internal_price"][:, single_s_fixed],
    "admm_replay_price": single_admm_replay["internal_price"][:, single_s_fixed],
})

display(single_price_df)

plt.figure(figsize=(11, 4))
plt.plot(single_price_df["t"], single_price_df["cagg_internal_price"], marker="o", label="CAGG dual price")
plt.plot(single_price_df["t"], single_price_df["admm_final_price"], marker="x", label="ADMM final price")
plt.plot(single_price_df["t"], single_price_df["P_RT"], linestyle="--", alpha=0.7, label="P_RT")
plt.plot(single_price_df["t"], single_price_df["P_PN"], linestyle="--", alpha=0.7, label="P_PN")
plt.title(f"Internal price comparison, scenario={single_s_fixed}")
plt.xlabel("time")
plt.ylabel("price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Individual Profit Comparison

This table compares each DER's profit under independent participation versus CAGG, DAgg, ADMM final iterate, and ADMM price replay. DAgg and ADMM columns include allocated imbalance adjustment, following the allocation style in `0527.ipynb`.

In [ ]:
display(single_individual_df)

profit_cols = [
    "individual_profit",
    "cagg_profit_at_central_price",
    "dagg_profit_with_imbalance",
    "admm_final_profit_with_imbalance",
    "admm_replay_profit_with_imbalance",
]

ax = single_individual_df.set_index("der")[profit_cols].plot(kind="bar", figsize=(14, 5))
ax.set_title("DER-level profit comparison")
ax.set_xlabel("DER")
ax.set_ylabel("profit")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

gain_cols = ["cagg_gain_vs_individual", "dagg_gain_vs_individual", "admm_final_gain_vs_individual", "admm_replay_gain_vs_individual"]
display(single_individual_df[["der", *gain_cols]])

In [ ]:
single_hist = single_admm["history"].copy()
display(single_hist.tail(10))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogy(single_hist["iteration"], single_hist["primal_residual"], label="primal")
axes[0].semilogy(single_hist["iteration"], single_hist["dual_residual"], label="dual")
axes[0].semilogy(single_hist["iteration"], single_hist["eps_primal"], "--", label="eps primal")
axes[0].semilogy(single_hist["iteration"], single_hist["eps_dual"], "--", label="eps dual")
axes[0].set_title("ADMM residuals")
axes[0].set_xlabel("iteration")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(single_hist["iteration"], single_hist["objective"], label="ADMM")
axes[1].axhline(single_centralized["expected_profit"], color="black", linestyle="--", label="centralized")
axes[1].set_title("Expected profit")
axes[1].set_xlabel("iteration")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].semilogy(single_hist["iteration"], single_hist["max_balance_error"], label="max balance error")
axes[2].semilogy(single_hist["iteration"], single_hist["mean_abs_balance_error"], label="mean abs balance error")
axes[2].set_title("Internal balance error")
axes[2].set_xlabel("iteration")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Price comparison is now included above with CAGG, DAgg, and ADMM columns.

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(price_df["t"], price_df["centralized_internal_price"], marker="o", label="centralized dual price")
plt.plot(price_df["t"], price_df["admm_internal_price"], marker="x", label="ADMM dual-update price")
plt.plot(price_df["t"], price_df["P_RT"], linestyle="--", alpha=0.7, label="P_RT")
plt.plot(price_df["t"], price_df["P_PN"], linestyle="--", alpha=0.7, label="P_PN")
plt.title(f"Scenario-dependent internal price, s={s_fixed}")
plt.xlabel("time")
plt.ylabel("price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5B. Single-Case Rho Sensitivity

This section runs rho sensitivity on the same single 10-DER instance above.

Default setup:

```text
N = SINGLE_N_DER
S = SINGLE_SCENARIOS
seed = SINGLE_SEED
rho values = [0.05, 0.1, 0.5, 1.0, 2.0]
```

Use this before the overnight experiment if you want to choose a reasonable `rho` for the full run. The centralized benchmark is solved once, then ADMM is rerun for each `rho` on the same data instance.

In [ ]:
SINGLE_RHO_VALUES = [0.05, 0.1, 0.5, 1.0, 2.0]
SINGLE_RHO_MAX_ITER = SINGLE_MAX_ITER
SINGLE_RHO_CSV = ROOT / "admm" / "single_case_rho_sensitivity.csv"

single_rho_df = run_rho_sensitivity(
    n_der=SINGLE_N_DER,
    scenarios=SINGLE_SCENARIOS,
    seed=SINGLE_SEED,
    rho_values=SINGLE_RHO_VALUES,
    max_iter=SINGLE_RHO_MAX_ITER,
    eps=EPS,
    level=LEVEL,
    output_csv=SINGLE_RHO_CSV,
    output_flag=OUTPUT_FLAG,
)

single_rho_df

In [ ]:
single_rho_cols = [
    "rho", "admm_iterations", "admm_runtime_seconds", "relative_gap",
    "max_balance_error", "mean_abs_balance_error", "price_rmse",
    "final_primal_residual", "final_dual_residual",
]
single_rho_df[single_rho_cols].sort_values("rho")

In [ ]:
plot_single_rho = single_rho_df.sort_values("rho")
fig, axes = plt.subplots(1, 4, figsize=(19, 4))

axes[0].plot(plot_single_rho["rho"], plot_single_rho["admm_runtime_seconds"], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("rho")
axes[0].set_ylabel("ADMM runtime (s)")
axes[0].set_title("Runtime")
axes[0].grid(alpha=0.3)

axes[1].plot(plot_single_rho["rho"], plot_single_rho["admm_iterations"], marker="o", color="tab:orange")
axes[1].set_xscale("log")
axes[1].set_xlabel("rho")
axes[1].set_ylabel("iterations")
axes[1].set_title("Iterations")
axes[1].grid(alpha=0.3)

axes[2].semilogy(plot_single_rho["rho"], plot_single_rho["max_balance_error"], marker="o", color="tab:green")
axes[2].set_xscale("log")
axes[2].set_xlabel("rho")
axes[2].set_ylabel("max balance error")
axes[2].set_title("Balance")
axes[2].grid(alpha=0.3)

axes[3].plot(plot_single_rho["rho"], plot_single_rho["price_rmse"], marker="o", color="tab:red")
axes[3].set_xscale("log")
axes[3].set_xlabel("rho")
axes[3].set_ylabel("price RMSE")
axes[3].set_title("Price RMSE")
axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.show()

Interpretation tip:

- Prefer `rho` values with low balance error and stable price RMSE.
- If runtime or iterations explode, that `rho` is not attractive for the overnight grid.
- If ADMM hits `max_iter` for every `rho`, compare final balance error and final residuals rather than only the convergence flag.

## 5. Overnight Paper-Scale Experiment

Target experiment:

- scenarios fixed at `S=100`,
- DER counts: `N = [10, 30, 50, 75, 100]`,
- seeds: `[1, 2, 3, 4, 5, 6, 7, 8, 9, 25]`,
- total cases: 50.

Each case solves the centralized benchmark and the ADMM benchmark. The runner saves one row after every completed case, so the experiment can be inspected or resumed.

Saved files:

- raw case results: `admm/admm_overnight_results.csv`
- seed-averaged summary: `admm/admm_overnight_seed_average.csv`

In [ ]:
OVERNIGHT_N_VALUES = [10, 30, 50, 75, 100]
OVERNIGHT_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 25]
OVERNIGHT_SCENARIOS = 100

# ADMM settings. RHO=1.0 was more stable in the small smoke tests.
OVERNIGHT_RHO = 1.0
OVERNIGHT_MAX_ITER = 200
OVERNIGHT_EPS = 1e-8
OVERNIGHT_LEVEL = "high"

RAW_RESULTS_CSV = ROOT / "admm" / "admm_overnight_results.csv"
AVG_RESULTS_CSV = ROOT / "admm" / "admm_overnight_seed_average.csv"

print("Total cases:", len(OVERNIGHT_N_VALUES) * len(OVERNIGHT_SEEDS))
print("Raw results:", RAW_RESULTS_CSV)
print("Averaged results:", AVG_RESULTS_CSV)

### Optional: Existing Progress Check

Run this cell before or after the overnight experiment to see how many cases have already been saved and which cases were completed most recently.

In [ ]:
if RAW_RESULTS_CSV.exists():
    existing_df = pd.read_csv(RAW_RESULTS_CSV)
    done = existing_df[["n_der", "scenarios", "seed"]].drop_duplicates()
    print(f"Completed cases: {len(done)} / {len(OVERNIGHT_N_VALUES) * len(OVERNIGHT_SEEDS)}")
    display(done.sort_values(["seed", "n_der"]).tail(20))
else:
    print("No existing overnight results yet.")

### Run Overnight Grid

This is the long-running experiment cell. Results are checkpointed after each completed case.

If the run is interrupted, execute this cell again with `skip_completed=True`; completed `(N, S=100, seed)` cases will be skipped.

In [ ]:
RUN_OVERNIGHT = False

if RUN_OVERNIGHT:
    overnight_df = run_overnight_scalability(
        n_values=OVERNIGHT_N_VALUES,
        seeds=OVERNIGHT_SEEDS,
        scenarios=OVERNIGHT_SCENARIOS,
        rho=OVERNIGHT_RHO,
        max_iter=OVERNIGHT_MAX_ITER,
        eps=OVERNIGHT_EPS,
        level=OVERNIGHT_LEVEL,
        output_csv=RAW_RESULTS_CSV,
        average_csv=AVG_RESULTS_CSV,
        skip_completed=True,
        time_limit=1200,
        time_limit_per_local=None,
        output_flag=OUTPUT_FLAG,
    )

    overnight_df.tail()
else:
    print("Overnight run is disabled. Set RUN_OVERNIGHT = True to launch the full grid.")

## 6. Load and Summarize Overnight Results

You can start from this section in the morning. It loads the saved raw CSV, recomputes seed averages, and prepares tables for runtime, iterations, communication burden, objective gap, balance error, and price error.

In [ ]:
raw_df = pd.read_csv(RAW_RESULTS_CSV)
avg_df = summarize_scalability_results(raw_df)
avg_df.to_csv(AVG_RESULTS_CSV, index=False, encoding="utf-8")

print("Raw cases:", len(raw_df))
print("Averaged rows:", len(avg_df))
display(avg_df)

In [ ]:
summary_cols = [
    "n_der", "scenarios", "seed_count",
    "centralized_expected_profit", "admm_expected_profit", "relative_gap",
    "centralized_runtime_seconds", "admm_runtime_seconds", "experiment_wall_seconds",
    "admm_iterations", "communication_scalars",
    "max_balance_error", "mean_abs_balance_error", "price_rmse",
]
summary_table = avg_df[summary_cols].sort_values("n_der")
summary_table

## 7. Runtime, Iteration, and Communication Figures

These figures are intended for paper revision diagnostics:

- centralized vs ADMM runtime,
- ADMM iteration count,
- communication burden,
- optimality gap,
- internal balance error,
- price RMSE relative to centralized dual prices.

In [ ]:
plot_df = summary_table.sort_values("n_der")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(plot_df["n_der"], plot_df["centralized_runtime_seconds"], marker="o", label="Centralized")
axes[0].plot(plot_df["n_der"], plot_df["admm_runtime_seconds"], marker="x", label="ADMM")
axes[0].set_xlabel("Number of DERs")
axes[0].set_ylabel("Runtime (s)")
axes[0].set_title("Runtime comparison, S=100")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(plot_df["n_der"], plot_df["admm_iterations"], marker="o", color="tab:orange")
axes[1].set_xlabel("Number of DERs")
axes[1].set_ylabel("Iterations")
axes[1].set_title("ADMM iterations")
axes[1].grid(alpha=0.3)

axes[2].plot(plot_df["n_der"], plot_df["communication_scalars"], marker="o", color="tab:green")
axes[2].set_xlabel("Number of DERs")
axes[2].set_ylabel("Scalars exchanged")
axes[2].set_title("Communication burden")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(plot_df["n_der"], 100 * plot_df["relative_gap"], marker="o")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xlabel("Number of DERs")
axes[0].set_ylabel("Relative gap (%)")
axes[0].set_title("Optimality gap")
axes[0].grid(alpha=0.3)

axes[1].semilogy(plot_df["n_der"], plot_df["max_balance_error"], marker="o")
axes[1].set_xlabel("Number of DERs")
axes[1].set_ylabel("Max balance error")
axes[1].set_title("Internal market balance")
axes[1].grid(alpha=0.3)

axes[2].plot(plot_df["n_der"], plot_df["price_rmse"], marker="o")
axes[2].set_xlabel("Number of DERs")
axes[2].set_ylabel("Price RMSE")
axes[2].set_title("ADMM price vs centralized dual price")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Seed-Level Variability

This section checks seed-level variation. The box plots can be useful for robustness discussion or appendix material.

In [ ]:
seed_view_cols = [
    "n_der", "seed", "relative_gap", "centralized_runtime_seconds",
    "admm_runtime_seconds", "admm_iterations", "communication_scalars",
    "max_balance_error", "price_rmse",
]
raw_df[seed_view_cols].sort_values(["n_der", "seed"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

raw_df.boxplot(column="admm_runtime_seconds", by="n_der", ax=axes[0])
axes[0].set_title("ADMM runtime by DER count")
axes[0].set_xlabel("Number of DERs")
axes[0].set_ylabel("Runtime (s)")

raw_df.boxplot(column="relative_gap", by="n_der", ax=axes[1])
axes[1].set_title("Relative gap by DER count")
axes[1].set_xlabel("Number of DERs")
axes[1].set_ylabel("Relative gap")

plt.suptitle("")
plt.tight_layout()
plt.show()

## 9. Rho Sensitivity Analysis

`rho` is an ADMM hyperparameter. It is not an economic price. It controls how strongly each local DER subproblem is pulled toward the coordinator consensus in each iteration.

Small `rho` can make balance correction slow. Large `rho` can force consensus aggressively but may make local optimization less smooth or increase dual residual movement. This section runs the same stochastic instance with multiple `rho` values and compares convergence, runtime, balance error, objective gap, and price RMSE.

Start with a small case before running `S=100` if you only want a quick check.

In [ ]:
RHO_SENSITIVITY_VALUES = [0.05, 0.1, 0.5, 1.0, 2.0]
RHO_SENSITIVITY_N = 10
RHO_SENSITIVITY_S = 100
RHO_SENSITIVITY_SEED = 1
RHO_SENSITIVITY_MAX_ITER = 200
RHO_SENSITIVITY_CSV = ROOT / "admm" / "admm_rho_sensitivity.csv"

rho_sensitivity_df = run_rho_sensitivity(
    n_der=RHO_SENSITIVITY_N,
    scenarios=RHO_SENSITIVITY_S,
    seed=RHO_SENSITIVITY_SEED,
    rho_values=RHO_SENSITIVITY_VALUES,
    max_iter=RHO_SENSITIVITY_MAX_ITER,
    eps=EPS,
    level=LEVEL,
    output_csv=RHO_SENSITIVITY_CSV,
    output_flag=OUTPUT_FLAG,
)

rho_sensitivity_df

In [ ]:
if "rho_sensitivity_df" not in globals() and RHO_SENSITIVITY_CSV.exists():
    rho_sensitivity_df = pd.read_csv(RHO_SENSITIVITY_CSV)

rho_cols = [
    "rho", "admm_iterations", "admm_runtime_seconds", "relative_gap",
    "max_balance_error", "mean_abs_balance_error", "price_rmse",
    "final_primal_residual", "final_dual_residual",
]
rho_sensitivity_df[rho_cols].sort_values("rho")

In [ ]:
plot_rho = rho_sensitivity_df.sort_values("rho")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(plot_rho["rho"], plot_rho["admm_runtime_seconds"], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("rho")
axes[0].set_ylabel("ADMM runtime (s)")
axes[0].set_title("Runtime sensitivity")
axes[0].grid(alpha=0.3)

axes[1].semilogy(plot_rho["rho"], plot_rho["max_balance_error"], marker="o")
axes[1].set_xscale("log")
axes[1].set_xlabel("rho")
axes[1].set_ylabel("Max balance error")
axes[1].set_title("Balance sensitivity")
axes[1].grid(alpha=0.3)

axes[2].plot(plot_rho["rho"], plot_rho["price_rmse"], marker="o")
axes[2].set_xscale("log")
axes[2].set_xlabel("rho")
axes[2].set_ylabel("Price RMSE")
axes[2].set_title("Price sensitivity")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()